# Automatidata project

## Course 3 - The Power of Statistics

You are a data professional in a data consulting firm called **Automatidata**. The current project for its newest client, the **New York City Taxi & Limousine Commission (New York City TLC)**, has reached its midpoint after completing a project proposal, Python coding work, and exploratory data analysis.

The new assignment is to analyze the relationship between **fare amount** and **payment type** by conducting an A/B test.


# Course 3 End-of-course project: Statistical analysis

In this activity, you will practice using statistics to analyze and interpret data. The activity covers descriptive statistics and hypothesis testing.

The purpose of this project is to demonstrate knowledge of how to prepare, create, and analyze A/B tests. The results should aim to identify ways to generate more revenue for taxi drivers.

> **Educational assumption:** For this exercise, assume that customers were randomly selected and divided into two groups:
>
> 1. Customers required to pay with credit card.
> 2. Customers required to pay with cash.
>
> Without this assumption, causal conclusions about the effect of payment method on fare amount would not be valid.

The goal is to determine whether customers who use credit cards pay higher fare amounts than customers who use cash.


## Project parts

1. Imports and data loading
2. Conduct EDA and hypothesis testing
3. Communicate insights with stakeholders


# Conduct an A/B test

## PACE stages

- Plan
- Analyze
- Construct
- Execute


## PACE: Plan

### Research question

**Is there a statistically significant difference in the average fare amount between customers who use credit cards and customers who use cash?**


### Task 1. Imports and data loading

Import the packages and libraries needed to compute descriptive statistics and conduct a hypothesis test.

Useful packages and functions include:

- `pandas`
- `numpy`
- `scipy.stats`
- `stats.ttest_ind(a, b, equal_var)`
- `mean()`


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats


In [ ]:
# Load dataset into dataframe
# Compatible with Google Colab, Jupyter y ejecución local
from pathlib import Path

DATA_FILENAME = "2017_Yellow_Taxi_Trip_Data.csv"

# Rutas locales posibles
local_files = [
    Path(DATA_FILENAME),
    Path("/content") / DATA_FILENAME,   # Google Colab
    Path("/mnt/data") / DATA_FILENAME,  # Entorno de respaldo
]

# Rutas públicas de GitHub
DATA_URLS = [
    (
        "https://raw.githubusercontent.com/"
        "Abraham123w/Google-Advanced-Data-Analytics-Professional-Certificate/"
        "main/3.%20The%20Power%20of%20Statistics/"
        "2017_Yellow_Taxi_Trip_Data.csv"
    ),
    (
        "https://raw.githubusercontent.com/"
        "Abraham123w/Google-Advanced-Data-Analytics-Professional-Certificate/"
        "main/2.%20Go%20Beyond%20the%20Numbers%20Translate%20Data%20into%20Insights/"
        "2017_Yellow_Taxi_Trip_Data.csv"
    ),
    (
        "https://raw.githubusercontent.com/"
        "Abraham123w/Google-Advanced-Data-Analytics-Professional-Certificate/"
        "main/1.%20Foundations%20of%20Data%20Science/"
        "2017_Yellow_Taxi_Trip_Data.csv"
    ),
]

taxi_data = None
data_source = None
errors = []

# 1. Buscar primero una copia local
for file_path in local_files:
    try:
        if file_path.exists() and file_path.stat().st_size > 100:
            taxi_data = pd.read_csv(file_path, index_col=0)
            data_source = str(file_path)
            break
    except Exception as error:
        errors.append(f"{file_path}: {error}")

# 2. Si no existe localmente, descargarla desde GitHub
if taxi_data is None:
    for url in DATA_URLS:
        try:
            candidate = pd.read_csv(url, index_col=0)

            if candidate.empty:
                raise ValueError("El archivo no contiene registros.")

            taxi_data = candidate
            data_source = url
            break

        except Exception as error:
            errors.append(f"{url}: {error}")

if taxi_data is None:
    details = "\n- ".join(errors)
    raise RuntimeError(
        "No fue posible cargar el dataset. Verifica que el repositorio sea público "
        "y que el archivo se llame exactamente 2017_Yellow_Taxi_Trip_Data.csv.\n\n"
        f"Intentos realizados:\n- {details}"
    )

print("Dataset cargado correctamente.")
print(f"Fuente: {data_source}")
print(f"Filas: {taxi_data.shape[0]:,}")
print(f"Columnas: {taxi_data.shape[1]}")

taxi_data.head()


## PACE: Analyze and Construct

Computing descriptive statistics allows us to understand the distribution of the data, identify potential outliers, and check for data-quality issues. It also helps determine whether the data meets assumptions required for an A/B test, such as normality or equal variances.


### Task 2. Data exploration

Use descriptive statistics to conduct exploratory data analysis.

The `payment_type` variable is encoded as:

- `1`: Credit card
- `2`: Cash
- `3`: No charge
- `4`: Dispute
- `5`: Unknown


In [ ]:
# Esto te dirá CUÁNTOS viajes hubo de cada tipo
taxi_data['payment_type'].value_counts()


You are interested in the relationship between payment type and the fare amount the customer pays. One approach is to compare the average fare amount for each payment type.


In [ ]:
taxi_data.groupby('payment_type')['fare_amount'].mean()


Based on the averages, customers who pay with credit cards appear to have a larger average fare amount than customers who pay with cash. However, this difference might arise from random sampling rather than representing a true population difference. A hypothesis test is needed to assess statistical significance.


### Task 3. Hypothesis testing

#### Null and alternative hypotheses

- **H₀:** There is no difference in the average fare amount between customers who use credit cards and customers who use cash.
- **Hₐ:** There is a difference in the average fare amount between customers who use credit cards and customers who use cash.

The steps are:

1. State the null and alternative hypotheses.
2. Choose a significance level.
3. Calculate the p-value.
4. Reject or fail to reject the null hypothesis.

The significance level is **5%** (`α = 0.05`). A two-sample Welch's t-test is used.


In [ ]:
# 1. Separar los datos en dos grupos: Tarjeta de Crédito (1) y Efectivo (2)
credit_card = taxi_data[taxi_data['payment_type'] == 1]['fare_amount']
cash = taxi_data[taxi_data['payment_type'] == 2]['fare_amount']

# 2. Realizar la prueba T (t-test)
# Usamos equal_var=False porque en la vida real las varianzas de gastos suelen
# ser distintas (Prueba de Welch)
full_test = stats.ttest_ind(a=credit_card, b=cash, equal_var=False)
full_test


In [ ]:
alpha = 0.05

print(f"Estadístico t: {full_test.statistic:.6f}")
print(f"Valor p: {full_test.pvalue:.12g}")

if full_test.pvalue < alpha:
    print(
        "Reject the null hypothesis: there is a statistically significant "
        "difference in average fare amount between credit-card and cash customers."
    )
else:
    print(
        "Fail to reject the null hypothesis: there is not enough evidence of a "
        "difference in average fare amount."
    )


Since the p-value is substantially lower than the 5% significance level, the null hypothesis is rejected. The full dataset provides evidence of a statistically significant difference in average fare amount between customers who pay with credit cards and those who pay with cash.


In [ ]:
# 1. Separar y filtrar los grupos originales
credit_card_all = taxi_data[taxi_data['payment_type'] == 1]['fare_amount']
cash_all = taxi_data[taxi_data['payment_type'] == 2]['fare_amount']

# 2. Tomar una muestra aleatoria (n=30 de cada grupo)
# Usamos random_state=42 para que el resultado sea reproducible
credit_card_sample = credit_card_all.sample(n=30, random_state=42)
cash_sample = cash_all.sample(n=30, random_state=42)

# 3. Realizar la prueba T con las muestras
sample_test = stats.ttest_ind(
    a=credit_card_sample,
    b=cash_sample,
    equal_var=False
)

sample_test


In [ ]:
print(f"Estadístico t de la muestra: {sample_test.statistic:.6f}")
print(f"Valor p de la muestra: {sample_test.pvalue:.6f}")

if sample_test.pvalue < alpha:
    print("Reject the null hypothesis.")
else:
    print(
        "Fail to reject the null hypothesis because the p-value "
        f"({sample_test.pvalue:.3f}) is greater than the 5% significance level."
    )


## PACE: Execute

### Task 4. Communicate insights with stakeholders

#### 1. What business insights can be drawn from the hypothesis test?

The main business insight is that there is a statistically significant difference in fare amounts between payment types when the full dataset is used. Customers paying with credit cards tend to have higher fare amounts than those paying with cash.

This suggests that encouraging credit-card use through incentives or app features could potentially increase overall revenue. However, the analysis does not prove that the payment method itself causes higher fares unless customers were genuinely assigned to payment groups at random.

#### 2. Why might this A/B test project not be realistic?

This project relies on observational historical data rather than a controlled experiment. In a true A/B test, participants would be randomly assigned to groups to reduce selection bias.

In the real dataset, customers selected their own payment method, which introduces confounding variables. For example, customers may be more likely to use cards for longer or more expensive trips. In addition, cash tips are not fully recorded, so comparisons involving total payment may be biased toward credit-card users.


## Conclusion

The full-data Welch's t-test indicates a statistically significant difference between the average fare amounts of credit-card and cash trips. The smaller random sample does not produce the same conclusion, illustrating how sample size affects statistical power and uncertainty.

Business decisions should therefore consider both statistical significance and study design. A properly randomized experiment would be required before making a causal claim that encouraging credit-card payments directly increases fare amounts.


**Congratulations! You have completed this lab.**
